# Pregunta 3 — Cámaras trampa con YOLO sobre CCT-20

Cuaderno reproducible que **orquesta el pipeline** y reimprime las tablas y
figuras clave del informe a partir de los JSON de resultados ya escritos por
cada paso.

- **Entrenamiento y evaluación** viven en los scripts de `paso1`..`paso7`; este
  cuaderno no reimplementa lógica, solo la invoca y lee sus resultados.
- El pipeline completo se corre con **un único comando** (celda siguiente). Ese
  comando, en su primera etapa, **descarga ~18 GB de imágenes de LILA BC**
  (Caltech Camera Traps) y el entrenamiento toma horas en GPU. Por eso aquí, por
  defecto, **no** se ejecuta el pipeline pesado: se muestra cómo invocarlo y se
  leen los JSON que ya están en disco.

| Entregable | Paso | Resultado que se lee aquí |
|---|---|---|
| #1 modelo YOLO | paso3 | `corridas/yolo11n_640/resumen.json` |
| #2 mAP por especie | paso4 | `paso4_evaluacion/informes/metricas.json` |
| #3 generalización + Grad-CAM | paso5 | `generalizacion.json`, `gradcam.json`, figuras |
| #4 robustez | paso5 | `robustez.json` |
| #5 monitoreo (cámara no vista) | paso6 | `actividad_trans_test.json` |
| #6 informe + comando único | paso7 | `informe/informe.pdf` |


## 1. Comando único (crear venv, instalar, ejecutar de extremo a extremo)

```bash
python -m venv .venv
.venv/Scripts/activate            # Windows; source .venv/bin/activate en Unix
pip install -r pregunta3/requirements.txt
python pregunta3/paso7_informe/ejecutar_todo.py
```

`ejecutar_todo.py` es idempotente: cada etapa se salta si su artefacto ya existe.
La celda siguiente **solo lista** las etapas y su estado (`--listar`), sin correr
nada pesado ni descargar el dataset.

In [ ]:
import json
import pathlib
import subprocess
import sys

# Raíz del repo, sin depender del directorio de trabajo.
RAIZ = pathlib.Path.cwd()
if not (RAIZ / "pregunta3").is_dir():
    RAIZ = RAIZ.parent
P3 = RAIZ / "pregunta3"
assert P3.is_dir(), f"no encuentro pregunta3/ desde {RAIZ}"
print("raíz del repo:", RAIZ)

# Listar las etapas del pipeline (no corre nada pesado).
try:
    salida = subprocess.run(
        [sys.executable, str(P3 / "paso7_informe" / "ejecutar_todo.py"), "--listar"],
        cwd=RAIZ, capture_output=True, text=True, timeout=120)
    print(salida.stdout or salida.stderr)
except Exception as exc:  # noqa: BLE001
    print("no se pudo listar (requiere el entorno instalado):", exc)


## 2. Cargar los JSON de resultados

Helper que lee un JSON de resultados y avisa (sin romper) si todavía no se
generó.

In [ ]:
def cargar(ruta_rel):
    ruta = P3 / ruta_rel
    if not ruta.is_file():
        print(f"(falta {ruta_rel}; corré ejecutar_todo.py para generarlo)")
        return None
    return json.loads(ruta.read_text(encoding="utf-8"))

metricas       = cargar("paso4_evaluacion/informes/metricas.json")
generalizacion = cargar("paso5_generalizacion/informes/generalizacion.json")
robustez       = cargar("paso5_generalizacion/informes/robustez.json")
gradcam        = cargar("paso5_generalizacion/informes/gradcam.json")
actividad_cis  = cargar("paso6_monitoreo/informes/actividad.json")
actividad_trans= cargar("paso6_monitoreo/informes/actividad_trans_test.json")


## 3. Entregable #2 — mAP por especie (`cis_test` vs `trans_test`)

In [ ]:
def ap(x, ancho=8):
    return (" " * (ancho - 3) + "n/d") if x is None else f"{x:>{ancho}.4f}"

if metricas:
    for conj in ("cis_test", "trans_test"):
        d = metricas[conj]
        print(f"=== {conj}: mAP@0.5 macro {d['map50']:.4f} | "
              f"ponderado por soporte {d['map50_ponderado_por_soporte']:.4f} ===")
        print(f"{'especie':10} {'cajas':>7} {'detec.':>8} {'AP@0.5':>8} {'AP@.5:.95':>10}")
        for nombre, v in sorted(d["por_clase"].items(), key=lambda kv: -kv[1]["soporte"]):
            print(f"{nombre:10} {v['soporte']:>7,} {v['predicciones']:>8,} "
                  f"{ap(v['ap50'])} {ap(v['ap50_95'], 10)}")
        if d.get("clases_sin_positivos"):
            print("  sin cajas anotadas (AP indefinida):",
                  ", ".join(d["clases_sin_positivos"]))
        print()


## 4. Entregable #3 — brecha de generalización `cis` vs `trans`

In [ ]:
if metricas:
    cis, trans = metricas["cis_test"]["map50"], metricas["trans_test"]["map50"]
    print(f"mAP@0.5  cis_test = {cis:.4f}   trans_test = {trans:.4f}   "
          f"brecha = {cis - trans:.4f}")
if generalizacion:
    print("\ncaída de AP@0.5 por especie (ubicaciones vistas -> nuevas):")
    print(f"{'especie':10} {'AP cis':>8} {'AP trans':>9} {'caída':>8}")
    for b in sorted(generalizacion["brechas"],
                    key=lambda b: (b["caida"] is None, -(b["caida"] or 0.0))):
        caida = "n/d" if b["caida"] is None else f"{b['caida']:.4f}"
        apt = "n/d" if b["ap_nueva"] is None else f"{b['ap_nueva']:.4f}"
        print(f"{b['nombre']:10} {ap(b['ap_vista'])} {apt:>9} {caida:>8}")


## 5. Entregable #4 — robustez ante tamaño del objeto

In [ ]:
if robustez:
    for conj, tramos in robustez["tamano"].items():
        print(f"=== {conj} ===")
        print(f"{'tramo':10} {'cajas':>7} {'mAP@0.5':>9}")
        for t in tramos:
            print(f"{t['tramo']:10} {t['cajas']:>7,} {t['map50']:>9.4f}")
        print()
    ocl = robustez.get("oclusion") or {}
    if ocl:
        print("oclusión sintética (mAP@0.5 por nivel de área tapada):")
        for modo, niveles in ocl.items():
            fila = "  ".join(f"{float(n):.0%}:{(v or {}).get('map50', float('nan')):.3f}"
                             for n, v in sorted(niveles.items(), key=lambda kv: float(kv[0])))
            print(f"  {modo:9} {fila}")


## 6. Entregable #5 — monitoreo sobre una cámara NO vista (`trans_test`)

La demostración principal del módulo es sobre `trans_test` (cámaras nunca vistas
en entrenamiento, el caso de despliegue real). `Error` es la diferencia relativa
contra el mismo reporte calculado desde las cajas anotadas a mano: la validación
del módulo. `Nocturnidad` es el control contra la historia natural conocida.

In [ ]:
def tabla_actividad(act, titulo):
    if not act:
        return
    err = {f["especie"]: f for f in act["comparacion_con_anotaciones"]}
    esf = act["esfuerzo"]
    print(f"=== {titulo} ({act['particion']}): {esf['camaras']} cámaras, "
          f"{esf['noches_camara']:,} noches-cámara ===")
    print(f"{'especie':10} {'eventos':>8} {'anotados':>9} {'error':>8} {'nocturn.':>9}")
    for e in act["especies"]:
        c = err.get(e["especie"], {})
        rel = c.get("error_relativo")
        rel_s = "n/d" if rel is None else f"{rel:+.0%}"
        print(f"{e['especie']:10} {e['eventos']:>8,} "
              f"{c.get('eventos_anotados', 0):>9,} {rel_s:>8} "
              f"{e['indice_nocturnidad']:>9.0%}")
    print()

tabla_actividad(actividad_trans, "DEMOSTRACIÓN principal, cámaras no vistas")
tabla_actividad(actividad_cis,  "comparación, cámaras vistas")


## 7. Entregable #3 — figuras Grad-CAM

El mapa de activación en una ubicación nunca vista: la evidencia que el modelo
acumula se reparte por el fondo y apenas toca al animal.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

figs = P3 / "paso7_informe" / "informe" / "figuras"
if not figs.is_dir():
    figs = P3 / "paso5_generalizacion" / "figuras"

nombres = ["trans_test_falso_positivo.jpg", "trans_test_acierto.jpg"]
disponibles = [figs / n for n in nombres if (figs / n).is_file()]
if disponibles:
    fig, ejes = plt.subplots(1, len(disponibles), figsize=(14, 6))
    if len(disponibles) == 1:
        ejes = [ejes]
    for ax, ruta in zip(ejes, disponibles):
        ax.imshow(mpimg.imread(str(ruta)))
        ax.set_title(ruta.name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(f"(no hay figuras en {figs}; corré el paso 5 para generarlas)")

if gradcam:
    for part in ("cis_test", "trans_test"):
        d = gradcam.get(part)
        if d:
            print(f"{part}: razón de masa mediana {d['razon_mediana']:.2f} "
                  f"(1.0 = el modelo no localiza mejor que el azar)")
